# ALDIMI - Predicciones

Modelo base para predecir Consumo_Diario con el dataset merged.

In [4]:
from pathlib import Path
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
BASE_DIR = Path.cwd()
if BASE_DIR.name == 'src':
    BASE_DIR = BASE_DIR.parent

DATA_PATH = BASE_DIR / 'data' / 'raw' / 'stock_raw.csv'
df = pd.read_csv(DATA_PATH)
print('Loaded:', df.shape)

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date'])

agg_map = {
    'Inventory_Level': 'mean'
}
if 'Units_Sold' in df.columns:
    agg_map['Units_Sold'] = 'sum'
if 'Supplier_Lead_Time_Days' in df.columns:
    agg_map['Supplier_Lead_Time_Days'] = 'mean'
if 'Reorder_Point' in df.columns:
    agg_map['Reorder_Point'] = 'mean'
if 'Order_Quantity' in df.columns:
    agg_map['Order_Quantity'] = 'sum'
if 'Demand_Forecast' in df.columns:
    agg_map['Demand_Forecast'] = 'mean'

if 'Inventory_Level' not in df.columns:
    raise ValueError('Inventory_Level not found in dataset')

daily = df.groupby('Date').agg(agg_map).sort_index()

for lag in [1, 7, 14]:
    daily[f'Inventory_Level_lag_{lag}'] = daily['Inventory_Level'].shift(lag)
    if 'Units_Sold' in daily.columns:
        daily[f'Units_Sold_lag_{lag}'] = daily['Units_Sold'].shift(lag)

daily['Inventory_Level_roll7'] = daily['Inventory_Level'].rolling(7).mean()
if 'Units_Sold' in daily.columns:
    daily['Units_Sold_roll7'] = daily['Units_Sold'].rolling(7).mean()

daily['Inventory_Level_t+7'] = daily['Inventory_Level'].shift(-7)
daily['Inventory_Level_t+14'] = daily['Inventory_Level'].shift(-14)

data = daily.dropna().copy()
print('Prepared time series rows:', len(data))

feature_cols = [c for c in data.columns if not c.startswith('Inventory_Level_t+')]

split_idx = int(len(data) * 0.8)
train = data.iloc[:split_idx]
test = data.iloc[split_idx:]

models = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=42)
}

def evaluate_model(model, X_train, y_train, X_test, y_test, label):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = mse ** 0.5
    print(f'{label} - MAE: {mae:.3f} RMSE: {rmse:.3f}')

for horizon in [7, 14]:
    target_col = f'Inventory_Level_t+{horizon}'
    print(f'\n=== Horizon t+{horizon} days ===')
    X_train = train[feature_cols]
    y_train = train[target_col]
    X_test = test[feature_cols]
    y_test = test[target_col]
    for name, model in models.items():
        evaluate_model(model, X_train, y_train, X_test, y_test, name)

Loaded: (91250, 15)
Prepared time series rows: 337

=== Horizon t+7 days ===


TypeError: got an unexpected keyword argument 'squared'